# Hard Q2 — 2025 洛杉磯 Eaton 山火燒灼面積與嚴重度地圖

## 分析策略

使用 **Sentinel-2 SR（`COPERNICUS/S2_SR_HARMONIZED`）** 計算 **dNBR（differenced Normalized Burn Ratio）**：
- 火前影像：2024-11-01 至 2025-01-06（火災發生前）
- 火後影像：2025-01-08 至 2025-02-15（Eaton 火災於 2025-01-07 爆發）
- NBR = (NIR − SWIR2) / (NIR + SWIR2) → dNBR = pre_NBR − post_NBR
- dNBR 分級（USGS 標準）：
  - < 0.1  : 未燒灼
  - 0.1–0.27 : 低嚴重度
  - 0.27–0.44 : 中低嚴重度
  - 0.44–0.66 : 中高嚴重度
  - > 0.66 : 高嚴重度

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

ee.Authenticate()
ee.Initialize(project='0')

In [2]:
# Eaton 火災中心附近 AOI（Altadena, CA）
aoi = ee.Geometry.Rectangle([-118.20, 34.10, -117.95, 34.25])

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit = 1 << 10
    cirrus_bit = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit).eq(0).And(
           qa.bitwiseAnd(cirrus_bit).eq(0))
    return image.updateMask(mask).divide(10000)

def get_median_s2(start, end):
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(aoi)
            .filterDate(start, end)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
            .map(mask_s2_clouds)
            .median())

pre_fire  = get_median_s2('2024-11-01', '2025-01-06')
post_fire = get_median_s2('2025-01-08', '2025-02-15')

print('火前/火後影像建立完成')

火前/火後影像建立完成


In [3]:
# NBR = (B8 - B12) / (B8 + B12)
def compute_nbr(image):
    return image.normalizedDifference(['B8', 'B12']).rename('NBR')

nbr_pre  = compute_nbr(pre_fire)
nbr_post = compute_nbr(post_fire)

# dNBR = pre - post（正值代表植被損失 → 燒灼）
dnbr = nbr_pre.subtract(nbr_post).rename('dNBR').clip(aoi)

In [4]:
# 嚴重度分級
severity = (
    dnbr.where(dnbr.lt(0.1),   0)   # 未燒灼
        .where(dnbr.gte(0.1).And(dnbr.lt(0.27)), 1)  # 低
        .where(dnbr.gte(0.27).And(dnbr.lt(0.44)), 2) # 中低
        .where(dnbr.gte(0.44).And(dnbr.lt(0.66)), 3) # 中高
        .where(dnbr.gte(0.66), 4)   # 高
).rename('severity').clip(aoi)

# 燒灼區 = 嚴重度 >= 1
burned_mask = dnbr.gte(0.1).clip(aoi)

In [5]:
# 計算燒灼總面積（平方公尺 → 英畝）
burned_area = burned_mask.multiply(ee.Image.pixelArea())
area_m2 = burned_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e10
).get('dNBR').getInfo()

area_acres = area_m2 / 4046.856
print(f'燒灼總面積: {area_m2:,.0f} m²  =  {area_acres:,.1f} 英畝')

燒灼總面積: 84,046,917 m²  =  20,768.4 英畝


In [6]:
# 各嚴重度面積（英畝）
severity_labels = {1: '低', 2: '中低', 3: '中高', 4: '高'}
rows = []
for cls, label in severity_labels.items():
    mask = severity.eq(cls)
    a = (mask.multiply(ee.Image.pixelArea())
             .reduceRegion(ee.Reducer.sum(), aoi, 10, maxPixels=1e10)
             .get('severity').getInfo())
    rows.append({'嚴重度': label, '面積 (英畝)': round(a / 4046.856, 1)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))

嚴重度  面積 (英畝)
  低   9808.8
 中低   4076.7
 中高   3832.5
  高   3050.6


In [7]:
# 地圖視覺化
Map = geemap.Map(center=[34.175, -118.075], zoom=12)

# 火後真彩色影像
Map.addLayer(post_fire.clip(aoi),
             {'bands': ['B4','B3','B2'], 'min': 0, 'max': 0.3},
             '火後影像（Sentinel-2 RGB）')

# dNBR
dnbr_vis = {'min': -0.2, 'max': 1.0,
            'palette': ['#00FF00','#FFFF00','#FFA500','#FF4500','#8B0000']}
Map.addLayer(dnbr, dnbr_vis, 'dNBR')

# 嚴重度分級
sev_vis = {'min': 0, 'max': 4,
           'palette': ['grey','#FFFF00','#FFA500','#FF4500','#8B0000']}
Map.addLayer(severity, sev_vis, '燒灼嚴重度 (0=未燒,1=低,2=中低,3=中高,4=高)')

Map

Map(center=[34.175, -118.075], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [8]:
print(f'\n====== Eaton 山火燒灼分析摘要 ======')
print(f'燒灼總面積: {area_acres:,.1f} 英畝')
print(df.to_string(index=False))
print('dNBR 閾值：≥0.1 判定為燒灼（USGS 標準）')


====== Eaton 山火燒灼分析摘要 ======
燒灼總面積: 20,768.4 英畝
嚴重度  面積 (英畝)
  低   9808.8
 中低   4076.7
 中高   3832.5
  高   3050.6
dNBR 閾值：≥0.1 判定為燒灼（USGS 標準）
